# graphArt_LM: the fine-tuned 1.5B on chosen prompts and data sets (evaluation only)

Runs the Phase 2 adapter (Qwen2.5-Coder-1.5B + LoRA) on the jobs in `EVALS` -- by default the
chain-of-simulation prompt (`cos`) on the 21 real torchvision samples. No training. About
20-60 minutes of one T4, capped at 2 hours.

**Before running**: *GPU T4 x2*, *Internet on*, and add the adapter folder
`qwen2.5-coder-1.5b-graphart-lora` as an input (*Add Input* -> your earlier upload, or
*Upload* the folder from `dist/qwen2.5-coder-1.5b-graphart-lora.zip`). Then
*Save Version -> Save & Run All (Commit)*.

**When it finishes**, download `graphart_hf_eval.zip` from the *Output* tab.

Generated by `scripts/make_kaggle_notebook.py --hf-eval`. Edit that script, not this file.

In [ ]:
# BASE_MODEL   the base the uploaded adapter was trained on (read from its adapter_config.json).
# EVALS        (prompt, data set, reply cap) jobs, run in order with the uploaded adapter.
#              Data sets: test (the stratified 40), torchvision, edits, rq6, large_test (80
#              library-style programs; give it a cap of ~7000 for the whole networks).
# TIME_BUDGET  the whole session; a job that would not fit is skipped, one in progress keeps
#              every reply finished so far (replies are cached one by one).
# EVAL_ARGS    extra flags for run_prompting.py, e.g. ["--limit", "2"] for a quick try.
# VALIDATE_ONLY True: only check the code arrived intact (works on a CPU session).
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"; EVALS = [("cos", "torchvision", 6500)]; TIME_BUDGET_HOURS = 2.0; EVAL_ARGS = []; VALIDATE_ONLY = False

## 1. Unpack the code, prompts and samples

In [ ]:
import base64, hashlib, io, json, os, shutil, subprocess, sys, tarfile, time, urllib.request
from pathlib import Path

START = time.time()
WORK = Path(os.environ.get("GRAPHART_WORK", "/kaggle/working"))
REPO = WORK / "graphArt_LM"
BUNDLE_SHA256 = "0ec8840066d6580f892a45f95f077a5f26e634260ceee917f4ba7f789564d229"
BUNDLE = (
    "..."
)
data = base64.b64decode(BUNDLE)
assert hashlib.sha256(data).hexdigest() == BUNDLE_SHA256, "bundle damaged -- regenerate the notebook"
REPO.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(data), mode="r:xz") as tar:
    tar.extractall(REPO)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print(sorted(str(p.relative_to(REPO)) for p in REPO.rglob("*") if p.is_file()))
if VALIDATE_ONLY:
    raise SystemExit("bundle intact -- VALIDATE_ONLY is set, so stopping here")
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)

## 2. Install the pinned libraries

In [ ]:
from importlib.metadata import PackageNotFoundError, version

PINS = {'transformers': '5.20.2', 'peft': '0.21.2', 'accelerate': '1.15.0'}

def pinned(package, wanted):
    try:
        return version(package) == wanted
    except PackageNotFoundError:
        return False

missing = [f"{p}=={v}" for p, v in PINS.items() if not pinned(p, v)]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# peft probes two optional quantisation libraries when it builds LoRA layers, and
# raises -- rather than skipping them -- if they are installed but older than it
# supports. Kaggle ships torchao 0.10 (peft wants >= 0.16), which killed a run. We
# use neither, so remove them if present.
for pkg in ("torchao", "gptqmodel"):
    try:
        print(f"removing {pkg} {version(pkg)} (peft rejects old versions; not used here)", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", pkg], check=True)
    except PackageNotFoundError:
        pass

subprocess.run([sys.executable, "-c", "import torch, transformers, peft; "
                "print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), "
                "'| transformers', transformers.__version__, '| peft', peft.__version__)"])

## 3. Download the base model

In [ ]:
# Its own step with a time limit: a stalled download inside training would otherwise
# sit until the session cap. ~3 GB; a few minutes on Kaggle's network.
FETCH = ("from huggingface_hub import snapshot_download; "
         f"print('downloaded to', snapshot_download({BASE_MODEL!r}))")
for attempt in (1, 2):
    try:
        subprocess.run([sys.executable, "-c", FETCH], check=True, timeout=1800)
        break
    except (subprocess.TimeoutExpired, subprocess.CalledProcessError) as exc:
        print(f"download attempt {attempt} failed: {exc}", flush=True)
else:
    raise SystemExit("could not download the base model -- stopping before any training")

## 4. Find the uploaded adapter

In [ ]:
import zipfile

GPU_ENV = dict(os.environ, CUDA_VISIBLE_DEVICES="0", HF_HUB_OFFLINE="1", PYTHONUNBUFFERED="1")
INPUT = Path(os.environ.get("GRAPHART_INPUT", "/kaggle/input"))

def remaining():
    return TIME_BUDGET_HOURS * 3600 - (time.time() - START)

def package():
    with zipfile.ZipFile(WORK / "graphart_hf_eval.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for p in (REPO / "results").rglob("*") if (REPO / "results").exists() else []:
            if p.is_file():
                z.write(p, p.relative_to(REPO))

adapter = None
for cfg in INPUT.rglob("adapter_config.json") if INPUT.exists() else []:
    if json.loads(cfg.read_text()).get("base_model_name_or_path") == BASE_MODEL \
            and (cfg.parent / "adapter_model.safetensors").exists():
        adapter = REPO / "lora" / cfg.parent.name
        shutil.copytree(cfg.parent, adapter, dirs_exist_ok=True)
        break
if adapter is None:
    raise SystemExit("no adapter for this base under /kaggle/input -- add it as an input (see the top)")
print(f"using the adapter {adapter.name}", flush=True)

## 5. Run the jobs

In [ ]:
for prompt, split, cap in EVALS:
    if remaining() < 600:
        print(f"time budget spent -- skipping {prompt} on {split}")
        continue
    cmd = [sys.executable, "-m", "pipeline.run_prompting", "--backend", "hf", "--model", str(adapter),
           "--prompt", prompt, "--split", split, *(["--max-tokens", str(cap)] if cap else []),
           *(["--stratify", "10"] if split == "test" else []), *EVAL_ARGS]
    print(f"\n##### {prompt} on {split}   ({remaining() / 3600:.1f} h left)", flush=True)
    try:
        subprocess.run(cmd, env=GPU_ENV, timeout=remaining())
    except subprocess.TimeoutExpired:
        print("stopped by the time budget -- the finished replies are kept")
    package()
for meta in sorted((REPO / "results").glob("*.meta.json")):
    m = json.loads(meta.read_text())
    print(f"{meta.stem:75s} n={m['n']} failed={m['failed']}")
print(f"done in {(time.time() - START) / 3600:.1f} h -- download graphart_hf_eval.zip")